In [ ]:
# 必要パッケージ
!pip install datasets==3.6.0
!pip install haystack-ai elasticsearch-haystack

In [ ]:
# SubjQA のサブセット確認
from datasets import get_dataset_config_names

domains = get_dataset_config_names("subjqa")
domains

In [ ]:
# Electronics ドメインのデータロード
from datasets import load_dataset

subjqa = load_dataset("subjqa", name="electronics")

In [ ]:
# answers 確認
print(subjqa["train"]["answers"][1])

In [ ]:
# pd に変換
import pandas as pd

dfs = {split: dset.to_pandas() for split, dset in subjqa.flatten().items()}

for split, df in dfs.items():
  print(f"Number of questions in {split}: {df['id'].nunique()}")

In [ ]:
# ランダムにサンプリング
qa_cols = ["title", "question", "answers.text",
           "answers.answer_start", "context"]
sample_df = dfs["train"][qa_cols].sample(2, random_state=7)
sample_df

In [ ]:
# 開始インデックスとアンサー長を使って、レビューから回答を抽出
start_idx = sample_df["answers.answer_start"].iloc[0][0]
end_idx = start_idx + len(sample_df["answers.text"].iloc[0][0])
sample_df["context"].iloc[0][start_idx:end_idx]

In [ ]:
# 質問の種類を確認
from matplotlib import pyplot as plt
counts = {}
question_types = ["What", "How", "Is", "Does", "Do", "Was", "Where", "Why"]

for q in question_types:
  counts[q] = dfs["train"]["question"].str.startswith(q).value_counts()[True]

pd.Series(counts).sort_values().plot.barh()
plt.title("Frequency of Question Types")
plt.show()

In [ ]:
# 例を見てみる
for question_type in ["How", "What", "Is"]:
  for question in (
      dfs["train"][dfs["train"].question.str.startswith(question_type)]
      .sample(n=3, random_state=42)['question']):
    print(question)

In [ ]:
# ベースモデルのトークナイザーロード
from transformers import AutoTokenizer

model_ckpt = "deepset/minilm-uncased-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
# サンプルをトークン化
question = "How much music can this hold?"
context = """An MP3 is about 1 MB/minute, so about 6000 hours depending on \
file size."""
inputs = tokenizer(question, context, return_tensors="pt")
print(inputs) # input_ids, token_type_ids, attention_mask からなる

In [ ]:
# input_ids をデコード
print(tokenizer.decode(inputs["input_ids"][0]))

In [ ]:
# 質問応答ヘッド
import torch
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(model_ckpt)

with torch.no_grad():
  outputs = model(**inputs)
print(outputs)

In [ ]:
# 開始/終了のロジット取得
start_logits = outputs.start_logits
end_logits = outputs.end_logits

In [ ]:
# ロジットの形状確認 (全て[1, 28])
print(f"Input IDs shape: {inputs.input_ids.size()}")
print(f"Start logits shape: {start_logits.size()}")
print(f"End logits shape: {end_logits.size()}")

In [ ]:
# ロジットを描画
import matplotlib.pyplot as plt
import numpy as np

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
start_scores = start_logits[0].detach().cpu().numpy()
end_scores = end_logits[0].detach().cpu().numpy()

x = np.arange(len(tokens))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 12), sharex=True)

ax1.bar(x, start_scores, label="Start Logits", color="skyblue", width=0.8)
ax1.set_title("Start Logits for Input Tokens")
ax1.set_ylabel("Logit Scores")
ax1.legend()
ax1.grid(True)
ax1.set_xlim([-0.5, len(tokens) - 0.5])

ax2.bar(x, end_scores, label="End Logits", color="lightcoral", width=0.8)
ax2.set_title("End Logits for Input Tokens")
ax2.set_xlabel("Tokens")
ax2.set_ylabel("Logit Scores")
ax2.legend()
ax2.grid(True)
ax2.set_xticks(x)
ax2.set_xticklabels(tokens, rotation=90)
ax2.set_xlim([-0.5, len(tokens) - 0.5])

plt.tight_layout()
plt.show()

In [ ]:
# argmax をとって回答を取得
start_idx = torch.argmax(start_logits)
end_idx = torch.argmax(end_logits)+1
answer_span = inputs["input_ids"][0][start_idx:end_idx]
answer = tokenizer.decode(answer_span)
print(f"Question: {question}")
print(f"Anser: {answer}")

In [ ]:
# question-answering で pipeline 化
from transformers import pipeline

pipe = pipeline("question-answering", model=model, tokenizer=tokenizer) # v5 では非推奨
pipe(question=question, context=context, topk=3)

In [ ]:
# 回答不可能な質問 (start, end ともに [CLS] が返ってきて欲しいが、[SEP] になってしまう)
pipe(question="Why is there no data?", context=context,
     handle_impossible_anser=True)

In [ ]:
# SubjQA における質問・コンテキストペアのトークン分布
token_lengths = []

for _, row in dfs["train"].iterrows():
    question = row["question"]
    context = row["context"]
    inputs = tokenizer(question, context, truncation=True)
    token_lengths.append(len(inputs["input_ids"]))

plt.figure(figsize=(10, 6))
plt.hist(token_lengths, bins=50, edgecolor='black')
plt.title('Distribution of Token Lengths for Question-Context Pairs')
plt.xlabel('Token Length')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
# スライディングウィンドウの適用例
example = dfs["train"].iloc[0][["question", "context"]]
tokenized_example = tokenizer(example["question"], example["context"],
                             return_overflowing_tokens=True, max_length=100,
                             stride=25)

# トークン数の確認
for idx, window in enumerate(tokenized_example["input_ids"]):
  print(f"Window #{idx} has {len(window)} tokens")

# デコード
for window in tokenized_example["input_ids"]:
  print(f"{tokenizer.decode(window)} \n")

In [ ]:
# Elasticsearch ダウンロード
url = "https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.17.10-linux-x86_64.tar.gz"
!wget -nc -q {url}
!tar -xzf elasticsearch-7.17.10-linux-x86_64.tar.gz

In [ ]:
# Elasticsearch 起動 (copilot 製)
# 1) 既存ES停止
!pkill -f org.elasticsearch.bootstrap.Elasticsearch || true

# 2) 必要なカーネル設定
!sysctl -w vm.max_map_count=262144

# 3) 所有者を daemon に変更
!chown -R daemon:daemon /content/elasticsearch-7.17.10

# 4) daemon で起動（cgroup回避オプション付き）
!su -s /bin/bash -c 'export ES_JAVA_OPTS="-Des.cgroups.hierarchy.override=/ -XX:-UseContainerSupport"; \
/content/elasticsearch-7.17.10/bin/elasticsearch \
  -Ediscovery.type=single-node \
  -Expack.security.enabled=false \
  -Expack.ml.enabled=false \
  -Ehttp.host=127.0.0.1 \
  -Etransport.host=127.0.0.1 \
  > /tmp/es.log 2>&1 &' daemon

# 5) 起動確認
!sleep 25
!tail -n 120 /tmp/es.log
!curl -s http://127.0.0.1:9200

In [ ]:
# Elasticsearch サーバーと通信テスト
## pretty=true で json に整形されて返ってくる
## 公式APIドキュメント：https://www.elastic.co/docs/reference/elasticsearch/rest-apis/common-options
!curl -X GET "localhost:9200/?pretty"

In [ ]:
# ドキュメントストアのインスタンス化
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore

# 文書埋め込みを返すようにしておく
document_store = ElasticsearchDocumentStore(hosts="http://localhost:9200")

In [ ]:
# Haystack でのインデックス化
from haystack.dataclasses import Document

for split, df in dfs.items():
  # 重複するレビュー除去
  docs = [Document(content=row["context"],
                   meta={"item_id": row["title"], "question_id": row["id"], "split": split})
          for _, row in df.drop_duplicates(subset="context").iterrows()]
  document_store.write_documents(docs) # エラー

print(f"Loaded {document_store.get_document_count()} documents")